# 02b – Kontrola baseline: generalizácia a artefakty

Rozšírenie resp. pokračovanie notebooku 02. Čistenie a rozdelenie podľa produktu sú rovnaké (seed 42 = rovnaké rozdelenie ako v 02).

- viac seedov: priemer ± smerodajná odchýlka cez rôzne rozdelenia produktov
- leave-one-generator-out (LOGO): train na dvoch generátoroch, test na treťom
- prah z validácie (maximum F1) namiesto 0.5; recall pri FPR 1 % (nezávisí od prahu)
- artefakty: typografická interpunkcia, HTML (`<br />`), dĺžka; ablácia s normalizáciou textu a so zarovnaním dĺžok
- znakové n-gramy ako druhý baseline (zachytávajú interpunkciu a HTML)
- analýza modelu: top príznaky (koeficienty LR), false negatives, false positives

Vstup modelu: iba text.

In [1]:
from pathlib import Path

DATA = Path("../../data/FraudSquad-Data/Amazon")
REAL_PATH = DATA / "amazon_real.csv"
SPAM_PATHS = {
    "llama3":    DATA / "reviews_from_llama3_8b.csv",
    "qwen_72b":  DATA / "reviews_from_qwen_72b.csv",
    "qwen_dsr1": DATA / "reviews_from_qwen_dsr1.csv",
}
COL = {"text": "reviewText", "rating": "star", "product": "product", "user": "user", "time": "reviewTime"}
RESULTS = Path("../../results")

SEEDS = [42, 43, 44]       # 42 = rovnaké rozdelenie ako v 02
VAL_FRAC = 0.15
TEST_FRAC = 0.15

DEDUP_REAL = True          # vyhodiť duplicitné texty v pravých recenziách
REAL_5STAR_ONLY = True     # spam je vždy 5★ -> porovnanie len s pravými 5★
DROP_NO_TEXT = True        # vyhodiť recenzie bez písmen/číslic

EQUAL_SPAM = True          # pri train na 2 generátoroch podvzorkovanie spamu na veľkosť jedného generátora
LEN_QUANTILES = (0.01, 0.99)   # pásmo dĺžok pre lenmatch: kvantily počtu slov spamu
TARGET_FPR = 0.01          # FPR pre metriku recall@FPR

RUN_ABLATION = True
ABLATION_KINDS = ["word", "char"]   # "char" má dlhší čas behu
ABLATION_SEEDS = SEEDS
TOP_K = 25                 # počet top príznakov vo výpise
N_EXAMPLES = 5             # počet false negatives / false positives vo výpise

In [2]:
import re, html, time, unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 300)

real = pd.read_csv(REAL_PATH)
spam = {name: pd.read_csv(p) for name, p in SPAM_PATHS.items()}
txt, rat, prod = COL["text"], COL["rating"], COL["product"]
GENS = list(spam)

EMOJI_RE = re.compile("[\U0001F1E6-\U0001F1FF\U0001F300-\U0001FAFF☀-➿⬀-⯿]")
JOINERS_RE = re.compile("[️‍]")

def has_no_text(t):
    rest = JOINERS_RE.sub("", EMOJI_RE.sub("", t))
    return re.search(r"\w", rest) is None

n0 = len(real)
real = real.dropna(subset=[txt])
if DEDUP_REAL:
    real = real.drop_duplicates(subset=[txt])
if REAL_5STAR_ONLY:
    real = real[real[rat] >= 5]
if DROP_NO_TEXT:
    real = real[~real[txt].astype(str).map(has_no_text)]
real = real.reset_index(drop=True)
print(f"real: {n0} -> {len(real)}")

for g in GENS:
    d = spam[g].dropna(subset=[txt])
    if DROP_NO_TEXT:
        d = d[~d[txt].astype(str).map(has_no_text)]
    spam[g] = d.reset_index(drop=True)
    print(f"{g}: {len(spam[g])}")

real: 86758 -> 55567
llama3: 2489
qwen_72b: 2495
qwen_dsr1: 2500


## 1. Interpunkcia a HTML
- podiel recenzií s typografickými znakmi (’ “ ” — …), HTML značkami a entitami, nie ASCII znakmi
- slovný TF-IDF tokenizuje `<br />` na token `br`; znakové n-gramy a subword tokenizéry zachytia aj typografické znaky
- normalizácia (`text_norm`): `html.unescape`, odstránenie `<br />` a vybraných HTML tagov, NFKC, typografické znaky -> ASCII ekvivalenty

In [3]:
CHECKS = {
    "’ ‘ typogr. apostrof":  r"[’‘]",
    "' rovný apostrof":      r"'",
    "“ ” typogr. úvodzovky": r"[“”]",
    "— – pomlčky":           r"[—–]",
    "… (jeden znak)":        r"…",
    "<br /> HTML":           r"(?i)<br\s*/?>",
    "&amp; a pod. entity":   r"&[a-z]+;",
}

def punct_stats(d):
    t = d[txt].astype(str)
    out = {k: t.str.contains(p, regex=True).mean() for k, p in CHECKS.items()}
    out["aspoň 1 ne-ASCII znak"] = t.map(lambda s: any(ord(c) > 127 for c in s)).mean()
    return out

rows = {"real (5★)": punct_stats(real)}
for g in GENS:
    rows[g] = punct_stats(spam[g])
print("Podiel recenzií, ktoré obsahujú daný znak:")
display(pd.DataFrame(rows).T.round(3))

Podiel recenzií, ktoré obsahujú daný znak:


,’ ‘ typogr. apostrof,' rovný apostrof,“ ” typogr. úvodzovky,— – pomlčky,… (jeden znak),<br /> HTML,&amp; a pod. entity,aspoň 1 ne-ASCII znak
real (5★),0.214,0.315,0.014,0.004,0.008,0.207,0.001,0.241
llama3,0.000,0.873,0.000,0.015,0.000,0.000,0.000,0.018
qwen_72b,0.354,0.690,0.001,0.250,0.000,0.000,0.000,0.499
qwen_dsr1,0.872,0.039,0.000,0.148,0.000,0.000,0.000,0.888


In [4]:
# Normalizácia: odstránenie HTML, typografické znaky -> ASCII, NFKC (… -> ..., NBSP -> medzera), zlúčenie medzier
BR_RE = re.compile(r"(?i)<br\s*/?>")
TAG_RE = re.compile(r"(?i)</?(?:p|a|b|i|u|div|span|strong|em)\b[^>]*>")
TRANS = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"', "—": "-", "–": "-"})

def normalize(s):
    s = html.unescape(str(s))
    s = BR_RE.sub(" ", s)
    s = TAG_RE.sub(" ", s)
    s = unicodedata.normalize("NFKC", s).translate(TRANS)
    return re.sub(r"\s+", " ", s).strip()

for d in [real, *spam.values()]:
    d["text_raw"] = d[txt].astype(str)
    d["text_norm"] = d["text_raw"].map(normalize)
    d["n_words"] = d["text_norm"].str.split().str.len()

ex = real[real["text_raw"].str.contains(BR_RE)].head(1)
if len(ex):
    print("pred:", ex["text_raw"].iloc[0][:200])
    print("po:  ", ex["text_norm"].iloc[0][:200])

rows = {"real (5★)": punct_stats(real.assign(**{txt: real["text_norm"]}))}
for g in GENS:
    rows[g] = punct_stats(spam[g].assign(**{txt: spam[g]["text_norm"]}))
print("\nPo normalizácii:")
display(pd.DataFrame(rows).T.round(3))

pred: I didn't need this app, but family was in town and they wanted to use the app.<br /><br />Was super easy to install and was the perfect addition to keep the siblings happy. They were able to watch wha
po:   I didn't need this app, but family was in town and they wanted to use the app. Was super easy to install and was the perfect addition to keep the siblings happy. They were able to watch what ever is o

Po normalizácii:


,’ ‘ typogr. apostrof,' rovný apostrof,“ ” typogr. úvodzovky,— – pomlčky,… (jeden znak),<br /> HTML,&amp; a pod. entity,aspoň 1 ne-ASCII znak
real (5★),0.0,0.524,0.0,0.0,0.0,0.0,0.0,0.028
llama3,0.0,0.873,0.0,0.0,0.0,0.0,0.0,0.003
qwen_72b,0.0,0.831,0.0,0.0,0.0,0.0,0.0,0.009
qwen_dsr1,0.0,0.894,0.0,0.0,0.0,0.0,0.0,0.020


**Výsledky:**
- `<br />`: 20,7 % pravých recenzií, 0 % spamu
- typografický apostrof: pravé 21,4 %, llama3 0 %, qwen_72b 35,4 %, qwen_dsr1 87,2 % -> vlastnosť konkrétneho generátora, nie LLM všeobecne
- pomlčky — –: pravé 0,4 %, llama3 1,5 %, qwen_72b 25,0 %, qwen_dsr1 14,8 %
- ne-ASCII: pravé 24,1 %, llama3 1,8 %, qwen_72b 49,9 %, qwen_dsr1 88,8 %
- po normalizácii: typografické znaky a HTML 0 % vo všetkých skupinách, ne-ASCII 0,3–2,8 % (emoji, diakritika)

In [5]:
# Pásmo dĺžok pre lenmatch: kvantily počtu slov všetkých spam recenzií
all_spam_len = pd.concat([spam[g]["n_words"] for g in GENS])
LO, HI = (int(round(x)) for x in all_spam_len.quantile(list(LEN_QUANTILES)))
print(f"pásmo dĺžok: {LO}–{HI} slov")
print(f"real v pásme: {real['n_words'].between(LO, HI).mean():.3f} ({real['n_words'].between(LO, HI).sum()} z {len(real)})")
for g in GENS:
    print(f"{g:10s} v pásme: {spam[g]['n_words'].between(LO, HI).mean():.3f}")

pásmo dĺžok: 38–81 slov
real v pásme: 0.317 (17636 z 55567)
llama3     v pásme: 0.994
qwen_72b   v pásme: 0.982
qwen_dsr1  v pásme: 0.970


**Výsledky:**
- pásmo: 38–81 slov
- v pásme: 31,7 % pravých recenzií, 97,0–99,4 % spamu
- `lenmatch=True` odstráni ~68 % pravých recenzií -> podiel spamu v teste stúpne približne 3×

## 2. Rozdelenie podľa produktu
- rovnaký postup ako v 02, samostatné rozdelenie pre každý seed
- rozdelenie závisí len od seedu, nie od filtrov -> všetky varianty ablácie majú rovnaké test produkty

In [6]:
spam_prods = sorted(set().union(*[set(spam[g][prod]) for g in GENS]))
other_prods = sorted(set(real[prod]) - set(spam_prods))

def split_map(seed):
    rng = np.random.default_rng(seed)
    m = {}
    for prods in (spam_prods, other_prods):
        p = np.array(prods)
        rng.shuffle(p)
        n = len(p); nt = int(n * TEST_FRAC); nv = int(n * VAL_FRAC)
        m.update({x: "test" for x in p[:nt]})
        m.update({x: "val" for x in p[nt:nt + nv]})
        m.update({x: "train" for x in p[nt + nv:]})
    return m

all_seeds = sorted(set(SEEDS) | set(ABLATION_SEEDS))
for s in all_seeds:
    m = split_map(s)
    for d in [real, *spam.values()]:
        d[f"split_{s}"] = d[prod].map(m)

for s in all_seeds:
    for d in [real, *spam.values()]:
        assert d[f"split_{s}"].notna().all()

s0 = SEEDS[0]
summary = {"real": real[f"split_{s0}"].value_counts()}
for g in GENS:
    summary[g] = spam[g][f"split_{s0}"].value_counts()
print(f"seed {s0}:")
display(pd.DataFrame(summary).fillna(0).astype(int))

seed 42:


,real,llama3,qwen_72b,qwen_dsr1
split_42,,,,
test,8307,375,375,375
train,39270,1739,1745,1750
val,7990,375,375,375


## 3. Pomocné funkcie
- `get_data`: pravé (0) + spam (1) pre dané generátory, časť (train / val / test), seed a nastavenia `norm`, `lenmatch`
- `fit`: TF-IDF + LR; prah z validácie tých istých generátorov ako train (held-out generátor sa nepoužíva ani pri výbere prahu)
- `evaluate`: PR-AUC, ROC-AUC, recall@FPR, precision / recall / F1 pri prahu z validácie, pre každý test generátor zvlášť
- `run`: všetky kombinácie train generátorov, typu príznakov, nastavení a seedov

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score, roc_curve,
                             precision_recall_curve, precision_recall_fscore_support)

def get_data(gens, split, seed, norm, lenmatch, equal_spam=False):
    col, sc = ("text_norm" if norm else "text_raw"), f"split_{seed}"
    r = real[real[sc] == split]
    ss = [spam[g][spam[g][sc] == split] for g in gens]
    if lenmatch:
        r = r[r["n_words"].between(LO, HI)]
        ss = [s[s["n_words"].between(LO, HI)] for s in ss]
    s = pd.concat(ss)
    if equal_spam and len(gens) > 1:
        s = s.sample(n=len(s) // len(gens), random_state=seed)
    X = np.concatenate([r[col].values, s[col].values])
    y = np.r_[np.zeros(len(r)), np.ones(len(s))]
    return X, y

def make_vec(kind):
    if kind == "word":
        return TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)
    if kind == "char":
        return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5,
                               sublinear_tf=True, max_features=300_000)
    raise ValueError(kind)

def best_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr * rc / np.maximum(pr + rc, 1e-12)
    return th[np.argmax(f1[:-1])]

def recall_at_fpr(y, p, fpr_max):
    fpr, tpr, _ = roc_curve(y, p)
    return tpr[fpr <= fpr_max].max()

def fit(gens, seed, kind="word", norm=False, lenmatch=False):
    Xtr, ytr = get_data(gens, "train", seed, norm, lenmatch, EQUAL_SPAM)
    vec = make_vec(kind)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(vec.fit_transform(Xtr), ytr)
    Xva, yva = get_data(gens, "val", seed, norm, lenmatch)
    thr = best_threshold(yva, clf.predict_proba(vec.transform(Xva))[:, 1])
    return vec, clf, thr

def evaluate(model, gens, seed, kind, norm, lenmatch):
    vec, clf, thr = model
    rows = []
    for g in GENS:
        Xte, yte = get_data([g], "test", seed, norm, lenmatch)
        p = clf.predict_proba(vec.transform(Xte))[:, 1]
        pr, rc, f1, _ = precision_recall_fscore_support(yte, p >= thr, average="binary", zero_division=0)
        rows.append(dict(train="+".join(gens), test=g, in_domain=g in gens, seed=seed,
                         kind=kind, norm=norm, lenmatch=lenmatch,
                         pr_auc=average_precision_score(yte, p), roc_auc=roc_auc_score(yte, p),
                         recall_at_fpr=recall_at_fpr(yte, p, TARGET_FPR),
                         f1=f1, precision=pr, recall=rc, thr=thr,
                         n_test_spam=int(yte.sum()), base_rate=yte.mean()))
    return rows

def run(trainsets, kinds, settings, seeds):
    rows, t0, n = [], time.time(), 0
    total = len(trainsets) * len(kinds) * len(settings) * len(seeds)
    for gens in trainsets:
        for kind in kinds:
            for norm, lenmatch in settings:
                for seed in seeds:
                    rows += evaluate(fit(gens, seed, kind, norm, lenmatch), gens, seed, kind, norm, lenmatch)
                    n += 1
                    print(f"\r{n}/{total} modelov, {time.time() - t0:.0f} s", end="")
    print()
    return pd.DataFrame(rows)

def mean_std(df, index, columns, value="pr_auc"):
    g = df.groupby(index + columns)[value]
    m = g.mean().unstack(columns).round(3)
    s = g.std().fillna(0).unstack(columns).round(3)
    return m.astype(str) + " ± " + s.astype(str)

## 4. Matica generátor × generátor
- riadky: train generátor, stĺpce: test generátor
- priemer ± std cez seedy
- text bez normalizácie (porovnateľné s 02)

In [8]:
singles = [[g] for g in GENS]
resA = run(singles, ["word"], [(False, False)], SEEDS)

for metric, name in [("pr_auc", "PR-AUC"), ("recall_at_fpr", f"Recall pri FPR {TARGET_FPR:.0%}"), ("f1", "F1 pri prahu z validácie")]:
    print(f"\n{name} (riadky = train, stĺpce = test):")
    display(mean_std(resA, ["train"], ["test"], metric))

9/9 modelov, 86 s

PR-AUC (riadky = train, stĺpce = test):


test,llama3,qwen_72b,qwen_dsr1
train,,,
llama3,0.995 ± 0.003,0.933 ± 0.004,0.975 ± 0.005
qwen_72b,0.95 ± 0.003,0.988 ± 0.001,0.96 ± 0.007
qwen_dsr1,0.851 ± 0.018,0.791 ± 0.015,0.996 ± 0.003



Recall pri FPR 1% (riadky = train, stĺpce = test):


test,llama3,qwen_72b,qwen_dsr1
train,,,
llama3,0.996 ± 0.003,0.9 ± 0.019,0.966 ± 0.011
qwen_72b,0.941 ± 0.003,0.993 ± 0.003,0.947 ± 0.009
qwen_dsr1,0.794 ± 0.02,0.731 ± 0.019,0.999 ± 0.002



F1 pri prahu z validácie (riadky = train, stĺpce = test):


test,llama3,qwen_72b,qwen_dsr1
train,,,
llama3,0.972 ± 0.006,0.795 ± 0.039,0.907 ± 0.008
qwen_72b,0.858 ± 0.013,0.949 ± 0.005,0.893 ± 0.012
qwen_dsr1,0.568 ± 0.051,0.403 ± 0.044,0.971 ± 0.009


**Výsledky:**
- PR-AUC na diagonále 0,988–0,996; std medzi seedmi 0,001–0,018
- hodnoty zodpovedajú 02 (napr. llama3 -> qwen_72b: 0,937 v 02, 0,933 +- 0,004 tu)
- najslabší transfer z qwen_dsr1: PR-AUC 0,851 (-> llama3), 0,791 (-> qwen_72b); opačný smer 0,975 (llama3 ->) a 0,960 (qwen_72b ->)
- qwen_dsr1 -> qwen_72b: recall@FPR 1 % 0,731, F1 0,403 -> F1 klesá výraznejšie ako metriky poradia; prah z validácie jedného generátora sa neprenáša na iný (posun kalibrácie)

## 5. Leave-one-generator-out
- train na dvoch generátoroch, test na všetkých troch
- `EQUAL_SPAM`: rovnaký počet spam recenzií v train ako pri jednom generátore
- porovnanie na held-out generátore: ten istý generátor / 1 iný generátor (priemer) / 2 iné generátory (LOGO)

In [9]:
logo = [[g for g in GENS if g != h] for h in GENS]
resB = run(logo, ["word"], [(False, False)], SEEDS)

print("PR-AUC (riadky = train, stĺpce = test):")
display(mean_std(resB, ["train"], ["test"], "pr_auc"))

offA = resA[~resA["in_domain"]].groupby("test")[["pr_auc", "recall_at_fpr", "f1"]].mean()
heldB = resB[~resB["in_domain"]].groupby("test")[["pr_auc", "recall_at_fpr", "f1"]].mean()
inA = resA[resA["in_domain"]].groupby("test")[["pr_auc", "recall_at_fpr", "f1"]].mean()
print("\nVýkon podľa toho, čo model videl pri tréningu (priemer cez seedy):")
display(pd.concat({"ten istý (diagonála)": inA,
                   "1 iný generátor": offA,
                   "2 iné generátory (LOGO)": heldB}, axis=1).round(3))

9/9 modelov, 93 s
PR-AUC (riadky = train, stĺpce = test):


test,llama3,qwen_72b,qwen_dsr1
train,,,
llama3+qwen_72b,0.991 ± 0.003,0.981 ± 0.001,0.978 ± 0.007
llama3+qwen_dsr1,0.99 ± 0.004,0.925 ± 0.003,0.995 ± 0.002
qwen_72b+qwen_dsr1,0.956 ± 0.009,0.977 ± 0.002,0.996 ± 0.002



Výkon podľa toho, čo model videl pri tréningu (priemer cez seedy):


ten istý (diagonála)                      1 iný generátor  \
                        pr_auc recall_at_fpr     f1          pr_auc   
test                                                                  
llama3                   0.995         0.996  0.972           0.901   
qwen_72b                 0.988         0.993  0.949           0.862   
qwen_dsr1                0.996         0.999  0.971           0.967   

                               2 iné generátory (LOGO)                       
          recall_at_fpr     f1                  pr_auc recall_at_fpr     f1  
test                                                                         
llama3            0.867  0.713                   0.956         0.945  0.888  
qwen_72b          0.816  0.599                   0.925         0.896  0.818  
qwen_dsr1         0.956  0.900                   0.978         0.976  0.932

**Výsledky (held-out generátor, PR-AUC):**

| test | 1 iný (priemer) | 1 iný (najlepší) | LOGO |
|---|---|---|---|
| llama3 | 0,901 | 0,950 | 0,956 |
| qwen_72b | 0,862 | 0,933 | 0,925 |
| qwen_dsr1 | 0,967 | 0,975 | 0,978 |

- LOGO > priemer jedného iného generátora; LOGO ≈ najlepší jeden iný generátor
- recall@FPR 1 %: llama3 0,867 -> 0,945, qwen_72b 0,816 -> 0,896, qwen_dsr1 0,956 -> 0,976
- F1 pri prahu z validácie: llama3 0,713 -> 0,888, qwen_72b 0,599 -> 0,818, qwen_dsr1 0,900 -> 0,932
- výkon na train generátoroch takmer nezmenený (napr. llama3 0,991 vs. 0,995)

## 6. Ablácia: normalizácia × lenmatch × slovné / znakové n-gramy
- LOGO train, 3 seedy, 72 modelov
- `norm`: normalizovaný text (sekcia 1)
- `lenmatch`: pravé recenzie aj spam len v pásme dĺžok spamu
- `word`: slovné 1–2-gramy; `char`: znakové 3–5-gramy (`char_wb`)

In [10]:
SETTINGS = [(False, False), (True, False), (False, True), (True, True)]
resC = None
if RUN_ABLATION:
    resC = run(logo, ABLATION_KINDS, SETTINGS, ABLATION_SEEDS)
    for label, mask in [("HELD-OUT generátor", ~resC["in_domain"]), ("generátory z tréningu", resC["in_domain"])]:
        print(f"\n{label}: priemer (std) cez generátory a seedy")
        t = resC[mask].groupby(["kind", "norm", "lenmatch"])[["pr_auc", "recall_at_fpr", "f1"]].agg(["mean", "std"])
        display(t.round(3))
    print("\nPR-AUC na held-out generátore, podľa generátora:")
    display(mean_std(resC[~resC["in_domain"]], ["kind", "norm", "lenmatch"], ["test"], "pr_auc"))
else:
    print("Ablácia vypnutá (RUN_ABLATION = False).")

72/72 modelov, 1114 s

HELD-OUT generátor: priemer (std) cez generátory a seedy


pr_auc        recall_at_fpr            f1       
                      mean    std          mean    std   mean    std
kind norm  lenmatch                                                 
char False False     0.948  0.025         0.923  0.038  0.869  0.037
           True      0.975  0.008         0.916  0.028  0.911  0.019
     True  False     0.951  0.022         0.929  0.034  0.877  0.041
           True      0.979  0.008         0.930  0.022  0.925  0.021
word False False     0.953  0.024         0.939  0.037  0.879  0.051
           True      0.982  0.006         0.938  0.027  0.917  0.027
     True  False     0.948  0.027         0.930  0.041  0.872  0.061
           True      0.980  0.006         0.925  0.030  0.914  0.026


generátory z tréningu: priemer (std) cez generátory a seedy


pr_auc        recall_at_fpr            f1       
                      mean    std          mean    std   mean    std
kind norm  lenmatch                                                 
char False False     0.990  0.007         0.988  0.011  0.949  0.017
           True      0.995  0.003         0.983  0.011  0.965  0.008
     True  False     0.986  0.007         0.982  0.010  0.940  0.016
           True      0.994  0.003         0.981  0.011  0.961  0.008
word False False     0.988  0.007         0.991  0.009  0.948  0.016
           True      0.996  0.003         0.990  0.009  0.969  0.012
     True  False     0.987  0.008         0.989  0.010  0.945  0.015
           True      0.995  0.003         0.988  0.010  0.966  0.013


PR-AUC na held-out generátore, podľa generátora:


test                        llama3       qwen_72b      qwen_dsr1
kind norm  lenmatch                                             
char False False     0.917 ± 0.018  0.959 ± 0.003  0.967 ± 0.006
           True      0.966 ± 0.005    0.982 ± 0.0  0.977 ± 0.001
     True  False     0.932 ± 0.015  0.946 ± 0.003  0.977 ± 0.004
           True      0.969 ± 0.004  0.981 ± 0.001  0.987 ± 0.001
word False False     0.956 ± 0.009  0.925 ± 0.003  0.978 ± 0.007
           True      0.981 ± 0.004  0.976 ± 0.003  0.988 ± 0.001
     True  False      0.95 ± 0.009  0.916 ± 0.001  0.978 ± 0.007
           True      0.979 ± 0.004  0.974 ± 0.002  0.987 ± 0.002

**Výsledky (held-out generátor):**
- recall@FPR 1 %: 0,916–0,939 vo všetkých 8 konfiguráciách; `norm` ani `lenmatch` ho výrazne nemenia (word: 0,939 -> 0,930 s `norm`, 0,938 s `lenmatch`)
- PR-AUC a F1 sú s `lenmatch=True` vyššie (word: 0,953 -> 0,982) v dôsledku vyššieho podielu spamu v teste, nie lepšieho poradia -> PR-AUC nie je porovnateľný medzi nastaveniami s rôznym podielom spamu
- `char` nie je lepší ako `word` (recall@FPR 0,923 vs. 0,939 bez úprav)
- `char` bez normalizácie, PR-AUC: llama3 0,917, qwen_72b 0,959 (word: 0,956 a 0,925); po normalizácii 0,932 a 0,946 -> znakový model využíva typografické znaky, ktorých výskyt sa medzi generátormi líši
- generátory z train: PR-AUC 0,986–0,996 vo všetkých konfiguráciách

## 7. Reziduálny signál v dĺžke
- rozhodovací strom (`max_depth=4`) len z počtu slov, všetky tri generátory spolu
- porovnanie bez zarovnania a so zarovnaním dĺžok

In [11]:
from sklearn.tree import DecisionTreeClassifier

rows = []
for lenmatch in [False, True]:
    for seed in SEEDS:
        def lens(split):
            X, y = get_data(GENS, split, seed, True, lenmatch)
            return np.array([len(t.split()) for t in X]).reshape(-1, 1), y
        Xtr, ytr = lens("train")
        Xte, yte = lens("test")
        tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=seed).fit(Xtr, ytr)
        p = tree.predict_proba(Xte)[:, 1]
        rows.append(dict(lenmatch=lenmatch, seed=seed, pr_auc=average_precision_score(yte, p),
                         roc_auc=roc_auc_score(yte, p), base_rate=yte.mean()))
display(pd.DataFrame(rows).groupby("lenmatch")[["pr_auc", "roc_auc", "base_rate"]].mean().round(3))

,pr_auc,roc_auc,base_rate
lenmatch,,,
False,0.375,0.885,0.118
True,0.401,0.673,0.295


**Výsledky:**
- bez `lenmatch`: PR-AUC 0,375 (podiel spamu 0,118), ROC-AUC 0,885
- s `lenmatch`: PR-AUC 0,401 (podiel spamu 0,295), ROC-AUC 0,673
- zarovnanie na pásmo signál z dĺžky znižuje, ale neodstraňuje; v rámci pásma je spam koncentrovaný, pravé recenzie sú rozložené rovnomernejšie

## 8. Top príznaky
- LR natrénovaná na všetkých troch generátoroch (prvý seed)
- 4 varianty: word / char × bez úprav / `norm` + `lenmatch`
- kladné koeficienty -> spam, záporné -> pravé recenzie
- príznaky vypísané cez `repr` (viditeľné medzery a špeciálne znaky)

In [12]:
def top_features(model, k=TOP_K):
    vec, clf, _ = model
    names, w = vec.get_feature_names_out(), clf.coef_[0]
    o = np.argsort(w)
    pos, neg = o[::-1][:k], o[:k]
    return pd.DataFrame({"-> spam": [repr(names[i]) for i in pos], "váha": w[pos].round(2),
                         "-> pravé": [repr(names[i]) for i in neg], "váha ": w[neg].round(2)})

for kind, norm, lenmatch in [("word", False, False), ("word", True, True), ("char", False, False), ("char", True, True)]:
    print(f"\n=== {kind}, norm={norm}, lenmatch={lenmatch}")
    display(top_features(fit(GENS, SEEDS[0], kind, norm, lenmatch)))


=== word, norm=False, lenmatch=False


,-> spam,váha,-> pravé,váha
0,'and the',8.14,'br',-7.82
1,'ve',7.74,'very',-6.87
2,'the',7.26,'this is',-4.98
3,'my',5.96,'great',-4.94
4,'plus',5.80,'good',-4.91
5,'lifesaver',5.65,'it is',-4.28
6,'without',5.51,'br br',-4.26
7,'absolutely',5.10,'will',-3.75
8,'game changer',5.03,'this',-3.72
9,'changer',4.93,'would',-3.65



=== word, norm=True, lenmatch=True


,-> spam,váha,-> pravé,váha
0,'and the',5.82,'very',-5.88
1,'ve',4.59,'great',-4.17
2,'game',4.44,'this is',-4.07
3,'app',4.17,'good',-3.78
4,'plus',4.01,'this',-3.58
5,'lifesaver',3.93,'you',-3.54
6,'me',3.88,'it is',-3.49
7,'my',3.70,'will',-3.23
8,'been',3.59,'nice',-3.19
9,'without',3.57,'would',-3.12



=== char, norm=False, lenmatch=False


,-> spam,váha,-> pravé,váha
0,' - ',6.27,'br ',-3.63
1,""" i'""",3.23,' />',-3.63
2,' my',3.05,'<br ',-3.63
3,' my ',2.79,'<br',-3.63
4,' me ',2.77,' very',-3.33
5,'my ',2.68,' ver',-3.20
6,'s! ',2.62,' ve',-3.11
7,"""'ve""",2.55,' it ',-3.09
8,"""'ve """,2.55,'it ',-2.69
9,' me',2.52,'use ',-2.43



=== char, norm=True, lenmatch=True


,-> spam,váha,-> pravé,váha
0,' - ',5.07,' it ',-2.84
1,""" i'""",2.03,' very',-2.58
2,'gam',1.99,'it ',-2.55
3,' gam',1.96,' ver',-2.40
4,' me ',1.83,' ve',-2.33
5,' my',1.81,'ll ',-2.12
6,'game',1.81,' you',-2.05
7,' en',1.80,'you',-2.04
8,"""'ve """,1.78,' thi',-1.97
9,"""'ve""",1.78,' yo',-1.95


**Výsledky:**
- spam (word): `and the`, `ve`, `my`, `me`, `plus`, `lifesaver`, `game changer`, `absolutely`, `thrilled`, `ensures`, `incredibly`, `fantastic`, `skeptical`, `expectations`
- pravé (word): `br` (najsilnejší príznak, −7,82), `br br`, `very`, `great`, `good`, `nice`, `this is`, `it is`, `cute`, `pretty`, `videoid`
- `videoid`: placeholder vloženého videa v Amazon recenziách (`[[VIDEOID:...]]`), normalizácia ho neodstraňuje
- po `norm` + `lenmatch`: `br` odstránený; medzi spam príznakmi pribudli `game` a `app` -> možný tematický artefakt (iné rozdelenie kategórií pri produktoch so spamom)
- char: najsilnejší spam príznak `' - '` (pomlčka s medzerami), aj po normalizácii; pravé: `<br` a jeho varianty

## 9. False negatives a false positives na held-out generátore
- word, `norm=True`, `lenmatch=True`, prvý seed
- false negatives: spam s najnižším skóre
- false positives: pravé recenzie s najvyšším skóre

In [13]:
for held in GENS:
    gens = [g for g in GENS if g != held]
    vec, clf, thr = fit(gens, SEEDS[0], "word", True, True)
    X, y = get_data([held], "test", SEEDS[0], True, True)
    p = clf.predict_proba(vec.transform(X))[:, 1]
    df = pd.DataFrame({"skóre": p.round(3), "label": y.astype(int), "text": X})
    print(f"\n=== held-out: {held} (prah z validácie {thr:.3f})")
    print("False negatives (spam, najnižšie skóre):")
    display(df[df["label"] == 1].nsmallest(N_EXAMPLES, "skóre"))
    print("False positives (pravé, najvyššie skóre):")
    display(df[df["label"] == 0].nlargest(N_EXAMPLES, "skóre"))


=== held-out: llama3 (prah z validácie 0.475)
False negatives (spam, najnižšie skóre):


,skóre,label,text
2576,0.169,1,"I'm not a professional musician, just a hobbyist who loves to play for fun. I was blown away by the quality of these strings - they brought out the best in my guitar and made it sound like a completely different instrument. The price is very reasonable too."
2756,0.232,1,"I was skeptical at first, thinking these lights might be too cheesy or gimmicky, but I was pleasantly surprised! They're actually really high-quality and fun. My friends loved how they changed colors and patterns in time with the music. We had a blast using them for our holiday party - would def..."
2700,0.234,1,"I was blown away by how easy it was to install these door knob covers! As a busy mom of three, I didn't have time to worry about whether they would fit or not. But with these Safety 1st Parent Grip Door Knob Covers, I just snapped them on and voila! My little ones can't open doors anymore. Love ..."
2831,0.243,1,"As a first-time parent, I was nervous about taking my little one to the beach or pool. But this baby carrier put those fears to rest! It's so easy to use and adjust, even with a squirmy baby in tow. The bright yellow color is fun and cheerful too - our whole family loves using it together. We'll..."
2828,0.249,1,"I bought this gift set as a baby shower present, and it was a huge hit! The mom-to-be loved all the goodies inside, from the soft washcloths to the cute onesie. And the best part? Everything is made with such high-quality materials - you can tell the designers put thought into every little detail."


False positives (pravé, najvyššie skóre):


,skóre,label,text
1256,0.834,0,"Overall, I would highly recommend this product to anyone looking for an effective way to ensure their little one is not harmed by sharp corners or edges when playing around the house. Not only does it provide superior safety measures but its stylish look helps blend into the surrounding décor wi..."
26,0.778,0,I love that this app helped me out with my animating skills and it also I helps everyone else out by having tutorials on almost every feature on the app. I really appreciate that the developers took the time to create this app. <3
809,0.734,0,"This is an ingenious child proof locking system, that works absolute wonders at keeping my inquisitive baby's prying limb safe! Not only are they fashionably designed with extra durable material, but the pinch proof feature offers an extra layer of safety so both toddler and I stay free from tea..."
797,0.689,0,"This stroller organizer is a must-have product for any new parent and has been such a lifesaver for me. It's a little bigger than I had expected, but it's perfect for our 9-month-old baby. It is a compact, yet durable and it all sorts of baby and non baby accessories like diapers, bottles, baby ..."
1956,0.658,0,"This cup is great for kids. Leak proof, easy to clean, dishwasher friendly. We've been using this for over a year now and it's been amazing every step of the way. Props for making a simple yet effective sipping device :)"



=== held-out: qwen_72b (prah z validácie 0.445)
False negatives (spam, najnižšie skóre):


,skóre,label,text
2890,0.068,1,"Just installed Lilly's Love hammock in my son's room yesterday, and it was shockingly simple! The hardware provided made it easy to set up without drilling holes everywhere. What a great idea to keep those cuddly friends off the floor while still allowing them to be part of his playtime fun. It ..."
2704,0.130,1,"These PTSLKHN nose pads have breathed new life into an old favorite pair of sunglasses! With regular use, they had become quite slippery on my nose until I found these silicone cushions. Now, even during long hikes or bike rides, they stay put comfortably and securely. The adhesive is very stron..."
2820,0.176,1,"Absolutely delighted with the Joy Gift Baby Basket! The keepsake box is perfect for storing baby's firsts-teeth, shoes, you name it! Every item inside is crafted from high-quality, soft materials; the plush toy and blanket are super comforting. It's a well-thought-out set that both new parents a..."
2916,0.197,1,"These Curlz are my savior when it comes to nutritious snacking! With my kid's active lifestyle, I need something quick yet wholesome - these organic snacks tick all the boxes. Sweet Potato & Cinnamon is his top pick; they're so tasty he doesn't even notice how many veggies he's eating!"
2609,0.206,1,"I absolutely love these guitar picks! The customization process was so easy; just uploaded my band's logo through the ""Customize now"" button, chose text color and font size, and voila-my very own personalized picks. They arrived in a neat pack of six with a professional finish. I can't get enoug..."


... (2 outputs left out)


=== held-out: qwen_dsr1 (prah z validácie 0.421)
False negatives (spam, najnižšie skóre):


,skóre,label,text
2928,0.174,1,"I purchased this keyboard piano for my 8-year-old niece, and she absolutely loves it! The keys are full size, which is perfect for learning proper technique. The 60 demo songs and 200 tones keep her entertained and motivated. The built-in stand and sheet music holder are thoughtful additions. It..."
2792,0.216,1,"These tumblers are amazing! My 2-year-old refuses to use sippy cups, but these are a hit. The straw is soft and comfortable, and the lid doesn't leak at all. The base sticks well to surfaces, preventing spills, but lifts off easily. They're also dishwasher safe, which is a huge convenience. The ..."
2781,0.248,1,"The ASC Elite Tablet Holder is an amazing product! It's slim, lightweight, and works perfectly with the rear-facing mirror. My kids love using it, and it's a great way to keep them occupied during long car rides. I would buy this again in a heartbeat!"
2627,0.295,1,"These corner guards are fantastic! They're easy to apply, hold up well, and are practically invisible once installed. My baby is always exploring, and these guards have prevented so many potential injuries. The quality is excellent, and they've made baby-proofing our home a breeze. I would buy t..."
2612,0.304,1,"I'm so happy with this RedThunder K10 combo! The wireless connection is fast and reliable, and the keyboard's mechanical feel is perfect for gaming. The LED backlighting is vibrant and customizable, which is a great bonus. The mouse is precise and comfortable, and the battery life is excellent. ..."


False positives (pravé, najvyššie skóre):


,skóre,label,text
26,0.890,0,I love that this app helped me out with my animating skills and it also I helps everyone else out by having tutorials on almost every feature on the app. I really appreciate that the developers took the time to create this app. <3
1256,0.845,0,"Overall, I would highly recommend this product to anyone looking for an effective way to ensure their little one is not harmed by sharp corners or edges when playing around the house. Not only does it provide superior safety measures but its stylish look helps blend into the surrounding décor wi..."
3,0.653,0,What an amazing app! Keeps little entertained for hours. Easy to install on tablet and extremely easy to navigate. Items for younger one of all ages (and fun for the inner child of us older ones too😉). It's a must have!
809,0.637,0,"This is an ingenious child proof locking system, that works absolute wonders at keeping my inquisitive baby's prying limb safe! Not only are they fashionably designed with extra durable material, but the pinch proof feature offers an extra layer of safety so both toddler and I stay free from tea..."
2448,0.615,0,"This strap has been an absolute game changer in our home. We have people ranging from 2 years to 35 years in our home and it makes trading off so easy. A twist or two and it's adjusted to the next person with ease. I personally hated playing our Oculus with the original strap, but this is so com..."


**Výsledky:**
- false negatives: recenzie s konkrétnou personou alebo situáciou („As a busy mom of three“, darček pre neter), s konkrétnymi parametrami a názvami produktov, s menším počtom typických LLM fráz
- false positives: texty v štýle marketingového opisu („superior safety measures“, „fashionably designed“, „works absolute wonders“)
- rovnaké pravé recenzie (index 26, 1256, 809, 797, 1956) sú medzi false positives pri viacerých held-out generátoroch; index 26 pri všetkých troch -> možný šum v labeloch triedy 0 (incentivized reviews, staršie generované texty)

In [14]:
RESULTS.mkdir(exist_ok=True)
parts = [resA.assign(experiment="matica"), resB.assign(experiment="logo")]
if resC is not None:
    parts.append(resC.assign(experiment="ablacia"))
out = pd.concat(parts, ignore_index=True)
out.to_csv(RESULTS / "baseline_02b.csv", index=False)
print("uložené:", RESULTS / "baseline_02b.csv", out.shape)

uložené: ../results/baseline_02b.csv (270, 17)


## V7sledky zhrnutie
- text-only detekcia na held-out generátore: recall@FPR 1 % ~0,93 (word, LOGO)
- testované artefakty (`<br />`, typografická interpunkcia, dĺžka) slovný model výrazne nevyužíva
- PR-AUC a F1 závisia od podielu spamu -> pri porovnaní nastavení s rôznym podielom spamu sa používa recall@FPR alebo ROC-AUC
- LOGO: výkon ≈ najlepší jeden iný generátor, vyšší ako priemer; menší pokles F1 pri prahu z validácie
- transfer z qwen_dsr1 je najslabší pri všetkých seedoch
- nekontrolované artefakty: téma/typ produktu (`game`, `app`), `videoid`, reziduálna dĺžka v rámci pásma
- opakované false positives naznačujú šum v labeloch pravých recenzií